In [ ]:
from langchain.tools import tool

@tool
def process_user_data(user_id: str, is_active: bool) -> str:
    """根据用户ID和活动状态, 更新用户记录

    Args:
        user_id (str): 用户唯一标识符
        is_active (bool): 用户是否处于活跃状态(True/False)
    """
    status = "激活" if is_active else "禁用"
    return f"用户ID: {user_id} 已被更新为 {status} 状态。"


# --- 1. 使用invoke方法调用工具
tool_input_dict = {
    "user_id": "12345",
    "is_active": True
}
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
console = Console()
table = Table(title="工具信息", show_header=False)
table.add_row("工具名称:", process_user_data.name)
table.add_row("工具描述:", process_user_data.description)
table.add_row("工具参数:", str(process_user_data.args_schema))

console.print(table)
console.print(Panel(f"调用工具: {process_user_data.name}\n输入参数: {tool_input_dict}", title="调用信息", expand=False))

                                工具信息                                 
┌───────────┬───────────────────────────────────────────────────────────┐
│ 工具名称: │ process_user_data                                         │
│ 工具描述: │ 根据用户ID和活动状态, 更新用户记录                        │
│           │                                                           │
│           │ Args:                                                     │
│           │     user_id (str): 用户唯一标识符                         │
│           │     is_active (bool): 用户是否处于活跃状态(True/False)    │
│ 工具参数: │ <class 'langchain_core.utils.pydantic.process_user_data'> │
└───────────┴───────────────────────────────────────────────────────────┘

╭──────────────────── 调用信息 ─────────────────────╮
│ 调用工具: process_user_data                       │
│ 输入参数: {'user_id': '12345', 'is_active': True} │
╰───────────────────────────────────────────────────╯

In [4]:
# 传入一个简单字符串
from langchain.tools import tool
@tool
def simple_greeting(name: str) -> str:
    """返回一个简单的问候语

    Args:
        name (str): 用户的名字
    """
    return f"你好, {name}!"

result_simple = simple_greeting.invoke({"name": "Alice"})
print(result_simple)  # 输出: 你好, Alice!

result_simple = simple_greeting.invoke("Alice")
print(result_simple)

你好, Alice!
你好, Alice!


In [6]:
# 传入一个简单非字符串
from langchain.tools import tool
@tool
def whatis(what: int) -> str:
    """返回一个简单的问候语

    Args:
        what (int): 要查询的数字
    """
    return f"{what}!"

result_simple = whatis.invoke({"what": 5})
print(result_simple)  # 输出: 5!
# ValidationError: result_simple = whatis.invoke(5)
# print(result_simple)  # 输出: 5!

5!


In [9]:
from langchain.tools import tool

@tool("exchange_rate_helper", description="用于进行人民币与美元之间的汇率换算。当用户询问金额转换时使用此工具")
def convert_currency(amount: float, to_currency: str) -> str:
    """执行基础的货币汇率换算(示例： 1USD = 7.2CNY)

    Args:
        amount (float): 金额
        to_currency (str): 目标汇率

    Returns:
        str: 转换后的值
    """
    rate = 7.2
    if to_currency.lower() == 'usd':
        result = amount / rate
        return f"{amount}CNY ≈ {result:.2f}USD"
    elif to_currency.lower == "cny":
        result = amount * rate
        return f"{amount}USD ≈ {result:.2f}CNY"
    else:
        return f"No suport for {to_currency}"
    
print(convert_currency.name)
print(convert_currency.description)
print(convert_currency.invoke({"amount": 100, "to_currency": "usd"}))

exchange_rate_helper
用于进行人民币与美元之间的汇率换算。当用户询问金额转换时使用此工具
100.0CNY ≈ 13.89USD


In [1]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.tools import tool

class WeatherRequest(BaseModel):
    city: str = Field(..., description="城市名称")
    unit: Literal["celsius", "fahrenheit"] = Field("celsius", description="温度单位")
    include_forecast: bool = Field(False, description="是否包含未来5天的天气预报")

@tool
def get_weather(city: str, unit: Literal["celsius", "fahrenheit"] = "celsius", include_forecast: bool = False) -> str:
    """获取指定城市的天气信息

    Args:
        city (str): 城市名称
        unit (Literal["celsius", "fahrenheit"], optional): 温度单位. Defaults to "celsius".
        include_forecast (bool, optional): 是否包含未来5天的天气预报. Defaults to False.

    Returns:
        str: 天气信息
    """
    # 模拟获取天气数据
    temperature = 25 if unit == "celsius" else 77
    forecast = "未来5天晴朗" if include_forecast else ""
    return f"{city}的当前温度为{temperature}°{unit[0].upper()}. {forecast}"
    
get_weather.invoke({"city": "北京", "unit": "celsius", "include_forecast": True})

'北京的当前温度为25°C. 未来5天晴朗'

In [ ]:
weather_schema = {
    "type": "object",
    "property": {
        "city": {"type": "string", "description": "城市名称"},
        "unit": {"type": "string", "enum": ["celsius", "fahrenheit"], "description": "温度单位"},
        "include_forecast": {"type": "boolean", "description": "是否包含未来5天的天气预报"}
    },
    "required": ["city", "unit", "include_forecast"]
}

from langchain.tools import tool

@tool(args_schema=WeatherRequest)
def get_weather(city: str, unit: Literal["celsius", "fahrenheit"] = "celsius", include_forecast: bool = False) -> str:
    """获取指定城市的天气信息

    Args:
        city (str): 城市名称
        unit (Literal["celsius", "fahrenheit"], optional): 温度单位. Defaults to "celsius".
        include_forecast (bool, optional): 是否包含未来5天的天气预报. Defaults to False.

    Returns:
        str: 天气信息
    """
    # 模拟获取天气数据
    temperature = 25 if unit == "celsius" else 77
    forecast = "未来5天晴朗" if include_forecast else ""
    return f"{city}的当前温度为{temperature}°{unit[0].upper()}. {forecast}"

In [6]:
from langchain.tools import tool, ToolRuntime
from dataclasses import dataclass

@dataclass
class UserContext:
    user_id: str = "GUEST_001"

@tool
def check_user_profile(runtime: ToolRuntime[UserContext]) -> str:
    """检查用户的个人资料信息

    Args:
        runtime (ToolRuntime[UserContext]): 工具运行时上下文, 包含用户信息

    Returns:
        str: 用户个人资料信息
    """
    current_user_id = runtime.context.user_id
    if current_user_id == "GUEST_001":
        return "当前用户为访客, 无法获取个人资料信息。"
    else:
        # 模拟获取用户资料信息
        return f"用户ID: {current_user_id}, 个人资料信息: [模拟数据]"

@tool
def get_message_count(runtime: ToolRuntime[UserContext]) -> str:
    """获取当前用户的消息数量

    Args:
        runtime (ToolRuntime[UserContext]): 工具运行时上下文, 包含用户信息

    Returns:
        int: 当前用户的消息数量
    """
    return f"当前用户的消息数量为: {len(runtime.state.get("messages", []))}"

mock_context = UserContext(user_id="USER_123")
mock_status = {"messages": []}

mock_runtime = ToolRuntime(
    context=mock_context, 
    state=mock_status, 
    config={}, 
    stream_writer=lambda x: None, 
    tool_call_id="mock_tool_call_id", 
    store=None
    )


from rich.console import Console
from rich.panel import Panel
console = Console()

result = check_user_profile.invoke({ "runtime": mock_runtime })
console.print(Panel(title="工具调用结果: check_user_profile", renderable=result, expand=False))

mock_runtime2 = ToolRuntime(
    context=None, 
    state={"messages": ["msg1", "msg2", "msg3"]}, 
    config={},
    stream_writer=lambda x: None,
    tool_call_id="mock_tool_call_id_2",
    store=None
)
result_mock_2 = get_message_count.invoke({ "runtime": mock_runtime2 })
console.print(Panel(title="工具调用结果: get_message_count", renderable=result_mock_2, expand=False))

╭───── 工具调用结果: check_user_profile ─────╮
│ 用户ID: USER_123, 个人资料信息: [模拟数据] │
╰────────────────────────────────────────────╯

╭─ 工具调用结果: get_message_count ─╮
│ 当前用户的消息数量为: 3           │
╰───────────────────────────────────╯